# Lezione 12 — Costruendo un chatbot (parte 2: il modello e il suo deployment)

**Obiettivi di oggi**
- Descrivere i concetti chiave di un modello di traduzione automatica neurale applicato alla conversazione.
- Addestrare un mini modello encoder-decoder.
- Generar respuestas e interactuar con el chatbot ya entrenado.

Consulta il [capitolo 9 del manuale](../manual/09_chatbot.md) per la spiegazione completa.

**Nota**: un chatbot real se entrena con millones de conversaciones y varios días de cómputo. Aquí usamos un dataset de juguete —una docena de preguntas y respuestas cortas— para que puedas ver y entrenar el pipeline completo en minutos, no para obtener un chatbot que converse con fluidez.

## 1. Un dataset giocattolo di domande e risposte

In [ ]:
import tensorflow as tf
import numpy as np

pares = [
    ("hola", "inicio hola que tal"),
    ("como estas", "inicio bien gracias"),
    ("que es tensorflow", "inicio es una libreria de deep learning"),
    ("que es una red neuronal", "inicio un modelo inspirado en el cerebro"),
    ("adios", "inicio hasta pronto"),
]

preguntas = [p[0] for p in pares]
respuestas = [p[1] for p in pares]
respuestas_objetivo = [r + " fin" for r in respuestas]  # la salida esperada incluye el token de fin

print(preguntas)
print(respuestas_objetivo)

## 2. Tokenizza domande e risposte con un vocabolario condiviso

In [ ]:
LONGITUD_MAXIMA = 8

tokenizador = tf.keras.layers.TextVectorization(
    max_tokens=200,
    output_sequence_length=LONGITUD_MAXIMA,
)
tokenizador.adapt(preguntas + respuestas_objetivo)

vocabulario = tokenizador.get_vocabulary()
tamano_vocabulario = len(vocabulario)
print("Tamaño del vocabulario:", tamano_vocabulario)
print("Vocabulario:", vocabulario)

## 3. Prepara le sequenze di addestramento

Il decodificatore viene addestrato con il "teacher forcing": riceve la risposta reale spostata di un token, e apprende a predire il successivo.


In [ ]:
entrada_encoder = tokenizador(preguntas).numpy()
secuencia_completa = tokenizador(respuestas_objetivo).numpy()

entrada_decoder = secuencia_completa[:, :-1]   # todo menos el último token
objetivo_decoder = secuencia_completa[:, 1:]   # todo menos el primero (desplazado)

print("Forma entrada encoder:", entrada_encoder.shape)
print("Forma entrada decoder:", entrada_decoder.shape)

## 4. Definisci il modello codificatore-decodificatore


In [ ]:
DIMENSION_EMBEDDING = 32
UNIDADES = 64

# Codificador: lee la pregunta y produce un estado que resume su significado
entrada_encoder_capa = tf.keras.Input(shape=(LONGITUD_MAXIMA,))
embedding_encoder = tf.keras.layers.Embedding(tamano_vocabulario, DIMENSION_EMBEDDING)(entrada_encoder_capa)
_, estado_h, estado_c = tf.keras.layers.LSTM(UNIDADES, return_state=True)(embedding_encoder)

# Decodificador: parte del estado del codificador y genera la respuesta token a token
entrada_decoder_capa = tf.keras.Input(shape=(LONGITUD_MAXIMA - 1,))
embedding_decoder = tf.keras.layers.Embedding(tamano_vocabulario, DIMENSION_EMBEDDING)(entrada_decoder_capa)
salida_decoder, _, _ = tf.keras.layers.LSTM(UNIDADES, return_sequences=True, return_state=True)(
    embedding_decoder, initial_state=[estado_h, estado_c]
)
salida_final = tf.keras.layers.Dense(tamano_vocabulario, activation="softmax")(salida_decoder)

modelo = tf.keras.Model([entrada_encoder_capa, entrada_decoder_capa], salida_final)
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

## 5. Addestra il modello


In [ ]:
modelo.fit([entrada_encoder, entrada_decoder], objetivo_decoder, epochs=200, verbose=0)
print("Entrenamiento terminado.")

## 6. Genera una risposta token per token (decodifica avida)

Un chatbot reale genera solitamente diverse risposte candidate ("beam search") e scegli la migliore. Qui usiamo la versione più semplice: a ogni passo, si sceglie sempre il token più probabile.


In [ ]:
def generar_respuesta(pregunta):
    entrada = tokenizador([pregunta]).numpy()
    secuencia_generada = [vocabulario.index("inicio")]

    for _ in range(LONGITUD_MAXIMA - 1):
        entrada_actual = tf.keras.preprocessing.sequence.pad_sequences(
            [secuencia_generada], maxlen=LONGITUD_MAXIMA - 1, padding="post"
        )
        prediccion = modelo.predict([entrada, entrada_actual], verbose=0)
        siguiente_token = int(np.argmax(prediccion[0, len(secuencia_generada) - 1]))
        if vocabulario[siguiente_token] == "fin":
            break
        secuencia_generada.append(siguiente_token)

    palabras = [vocabulario[t] for t in secuencia_generada[1:]]  # sin el token "inicio"
    return " ".join(palabras)

for pregunta in ["hola", "como estas", "que es tensorflow"]:
    print(f"Tú: {pregunta}")
    print(f"Bot: {generar_respuesta(pregunta)}")
    print()

## Riepilogo

- El codificador comprime la pregunta en un estado; el decodificador genera la respuesta token a token a partir de ese estado.
- Entrenar con "teacher forcing" significa alimentar al decodificador con la respuesta real desplazada un token, no con sus propias predicciones.
- Con un dataset de juguete el chatbot memoriza las respuestas vistas; con millones de conversaciones reales, generalizaría a preguntas nuevas.

Continua con il [notebook de práctica di questa lezione](../notebooks_practica/clase12_practica_chatbot_modelo.ipynb).